# Fast Gemma GGUF Inference on NVIDIA T4 (Google Colab)
Ensure your Colab runtime is set to **T4 GPU** (Runtime > Change runtime type > T4 GPU).

### Step 1: Rapid Precompiled Wheel Installation
Using prebuilt CUDA wheels bypasses the lengthy 10-15 minute compilation step.

In [ ]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install huggingface_hub fastapi uvicorn pyngrok nest_asyncio pydantic requests

### Step 2: System CUDA & Dynamic Linker Configuration
Register Python-installed NVIDIA CUDA libraries with the dynamic linker cache (`ldconfig`) to prevent runtime loading issues.

In [ ]:
# Register NVIDIA library directories with dynamic linker
!find /usr/local/lib/python*/dist-packages/nvidia/ -type d -name "lib" > /etc/ld.so.conf.d/nvidia-cuda.conf 2>/dev/null || true
!ldconfig

### Step 3: Download & Load Model into GPU

In [ ]:
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

print("System CUDA configuration is intact.")

REPO_ID = "ggml-org/gemma-4-12B-it-GGUF"
FILENAME = "gemma-4-12B-it-Q4_0.gguf"

print(f"Downloading {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir="./models",
    local_dir_use_symlinks=False
)
print(f"Model downloaded and saved to local disk: {model_path}")

print("Loading model layers directly to GPU...")
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=8192,
    n_batch=512,
    verbose=False
)
print("\nModel is fully loaded into GPU and ready for queries!")

### Step 4: Local Interactive Chat (Optional)

In [ ]:
import time

print("Chat loop initialized! Type 'exit' or 'quit' to stop.\n" + "=" * 50)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

while True:
    user_input = input("\nYou: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("Stopping chat loop. Goodbye!")
        break

    if not user_input.strip():
        continue

    messages.append({"role": "user", "content": user_input})
    if len(messages) > 16:
        messages = [messages[0]] + messages[-12:]
    print("\nAI: ", end="", flush=True)

    response_stream = llm.create_chat_completion(
        messages=messages,
        max_tokens=4096,
        temperature=0.7,
        top_p=0.9,
        stream=True
    )

    assistant_response = ""
    token_count = 0
    start_time = None

    for chunk in response_stream:
        if "choices" in chunk and len(chunk["choices"]) > 0:
            delta = chunk["choices"][0].get("delta", {})
            if "content" in delta:
                token = delta["content"]
                if token:
                    if start_time is None:
                        start_time = time.time()
                    token_count += 1
                    assistant_response += token
                    print(token, end="", flush=True)

    end_time = time.time()
    print()

    if start_time and token_count > 0:
        elapsed_time = end_time - start_time
        tokens_per_second = token_count / elapsed_time
        print(f"[Speed: {tokens_per_second:.2f} tok/sec | Generated: {token_count} tokens | Time: {elapsed_time:.2f}s]")
    else:
        print("[Speed: 0.00 tok/sec (Empty or instant response)]")

    messages.append({"role": "assistant", "content": assistant_response})

### Step 5: Expose as API Server via ngrok
Turn this Colab session into a public streaming API server and connect remotely using `alphacode` or `terminal.py` on your computer!
Get your free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Any, Dict
from pyngrok import ngrok, conf
import getpass
import time
import uuid
import json

# Set ngrok token
ngrok_token = os.getenv('NGROK_AUTHTOKEN', '').strip()
if not ngrok_token:
    ngrok_token = getpass.getpass('Paste your ngrok authtoken here: ').strip()

if ngrok_token:
    conf.get_default().auth_token = ngrok_token

app = FastAPI(title="Colab Gemma GGUF API")

class ChatMessage(BaseModel):
    role: str
    content: Optional[str] = ''
    tool_calls: Optional[List[dict]] = None
    tool_call_id: Optional[str] = None
    class Config:
        extra = 'allow'

class ChatRequest(BaseModel):
    model: Optional[str] = None
    messages: List[ChatMessage]
    max_tokens: Optional[int] = Field(default=2048)
    temperature: Optional[float] = Field(default=0.7)
    top_p: Optional[float] = Field(default=0.9)
    stream: Optional[bool] = Field(default=True)
    tools: Optional[List[dict]] = None
    tool_choice: Optional[Any] = None
    class Config:
        extra = 'allow'

def fit_messages_to_context(model, messages, max_tokens, n_ctx=8192):
    safety_margin = 96
    token_budget = max(512, n_ctx - max_tokens - safety_margin)
    def estimate_tokens(msgs):
        try:
            joined = '\n'.join(str(m.get('content', '')) for m in msgs)
            return len(model.tokenize(joined.encode('utf-8', errors='ignore')))
        except Exception:
            return sum(len(str(m.get('content', ''))) for m in msgs) // 3
    if len(messages) <= 2:
        return messages
    if estimate_tokens(messages) <= token_budget:
        return messages
    pruned = [dict(m) for m in messages]
    for i in range(1, len(pruned) - 2):
        content = pruned[i].get('content', '')
        if isinstance(content, str) and len(content) > 1200:
            pruned[i]['content'] = content[:500] + '\n...[intermediate output truncated for context window]...\n' + content[-500:]
    if estimate_tokens(pruned) <= token_budget:
        return pruned
    system_msg = pruned[0]
    initial_goal = pruned[1] if len(pruned) > 1 else None
    recent_msgs = pruned[2:]
    while len(recent_msgs) > 2 and estimate_tokens([system_msg] + ([initial_goal] if initial_goal else []) + recent_msgs) > token_budget:
        recent_msgs.pop(0)
    final_msgs = [system_msg]
    if initial_goal and initial_goal not in recent_msgs:
        final_msgs.append(initial_goal)
    final_msgs.extend(recent_msgs)
    return final_msgs

def enforce_autonomous_system_prompt(raw_messages: List[dict]) -> List[dict]:
    messages = [dict(m) for m in raw_messages]
    tool_prompt = (
        '\nYou are an autonomous terminal agent running inside Alphacode CLI. '
        'You have full authorization to execute system commands and install packages via tools. '
        'When requested to execute commands or install software, invoke the bash tool or output commands directly without refusal.'
    )
    if messages and messages[0].get('role') == 'system':
        if 'autonomous terminal agent' not in messages[0].get('content', ''):
            messages[0]['content'] = messages[0].get('content', '') + '\n' + tool_prompt
    else:
        messages.insert(0, {'role': 'system', 'content': tool_prompt})
    return messages

@app.get('/health')
def health_check():
    return {'status': 'ok', 'model': FILENAME}

@app.get('/v1/models')
def list_models():
    curr_time = int(time.time())
    return {
        'object': 'list',
        'data': [
            {'id': FILENAME, 'object': 'model', 'created': curr_time, 'owned_by': 'local'},
            {'id': 'gemma-4-12B-it', 'object': 'model', 'created': curr_time, 'owned_by': 'local'},
            {'id': 'gemma-4-12b-it', 'object': 'model', 'created': curr_time, 'owned_by': 'local'}
        ]
    }

@app.post('/chat')
def chat_endpoint(request: ChatRequest):
    if 'llm' not in globals() or llm is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    raw_messages = [msg.model_dump(exclude_none=True) for msg in request.messages]
    requested_max_tokens = request.max_tokens or 2048
    formatted_messages = fit_messages_to_context(llm, raw_messages, requested_max_tokens, n_ctx=8192)
    if request.stream:
        def token_generator():
            try:
                res_stream = llm.create_chat_completion(
                    messages=formatted_messages,
                    max_tokens=requested_max_tokens,
                    temperature=request.temperature,
                    top_p=request.top_p,
                    stream=True
                )
                for chunk in res_stream:
                    if 'choices' in chunk and len(chunk['choices']) > 0:
                        delta = chunk['choices'][0].get('delta', {})
                        token = delta.get('content', '')
                        if token:
                            yield token
            except Exception as e:
                yield f'\n[Inference Error: {str(e)}]'
        return StreamingResponse(token_generator(), media_type='text/plain')
    else:
        try:
            res = llm.create_chat_completion(
                messages=formatted_messages,
                max_tokens=requested_max_tokens,
                temperature=request.temperature,
                top_p=request.top_p,
                stream=False
            )
            return {'response': res['choices'][0]['message'].get('content', '')}
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

@app.post('/v1/chat/completions')
def openai_chat_completions(request: ChatRequest):
    if 'llm' not in globals() or llm is None:
        raise HTTPException(status_code=503, detail='Model not loaded')
    raw_messages = [msg.model_dump(exclude_none=True) for msg in request.messages]
    requested_max_tokens = request.max_tokens or 2048
    messages_with_system = enforce_autonomous_system_prompt(raw_messages)
    formatted_messages = fit_messages_to_context(llm, messages_with_system, requested_max_tokens, n_ctx=8192)
    completion_id = f'chatcmpl-{uuid.uuid4().hex[:12]}'
    model_id = request.model or FILENAME
    created_ts = int(time.time())
    call_kwargs = {
        'messages': formatted_messages,
        'max_tokens': requested_max_tokens,
        'temperature': request.temperature,
        'top_p': request.top_p
    }
    if request.tools:
        call_kwargs['tools'] = request.tools
    if request.tool_choice is not None:
        call_kwargs['tool_choice'] = request.tool_choice
    if request.stream:
        def sse_generator():
            try:
                res_stream = llm.create_chat_completion(stream=True, **call_kwargs)
                for chunk in res_stream:
                    if 'choices' in chunk and len(chunk['choices']) > 0:
                        choice = chunk['choices'][0]
                        delta = choice.get('delta', {})
                        delta_payload = {}
                        if 'content' in delta and delta['content'] is not None:
                            delta_payload['content'] = delta['content']
                        if 'tool_calls' in delta and delta['tool_calls']:
                            delta_payload['tool_calls'] = delta['tool_calls']
                        payload = {
                            'id': completion_id,
                            'object': 'chat.completion.chunk',
                            'created': created_ts,
                            'model': model_id,
                            'choices': [{
                                'index': 0,
                                'delta': delta_payload,
                                'finish_reason': choice.get('finish_reason')
                            }]
                        }
                        yield f'data: {json.dumps(payload)}\n\n'
                yield 'data: [DONE]\n\n'
            except Exception as e:
                err_payload = {'error': str(e)}
                yield f'data: {json.dumps(err_payload)}\n\n'
                yield 'data: [DONE]\n\n'
        return StreamingResponse(sse_generator(), media_type='text/event-stream')
    else:
        try:
            res = llm.create_chat_completion(stream=False, **call_kwargs)
            choice = res['choices'][0]
            msg = choice.get('message', {})
            resp_msg = {'role': 'assistant', 'content': msg.get('content', '')}
            if 'tool_calls' in msg and msg['tool_calls']:
                resp_msg['tool_calls'] = msg['tool_calls']
            return {
                'id': completion_id,
                'object': 'chat.completion',
                'created': created_ts,
                'model': model_id,
                'choices': [{
                    'index': 0,
                    'message': resp_msg,
                    'finish_reason': choice.get('finish_reason', 'stop')
                }],
                'usage': res.get('usage', {'prompt_tokens': -1, 'completion_tokens': -1, 'total_tokens': -1})
            }
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

# Open ngrok tunnel
PORT = 8000
ngrok.kill()
tunnel = ngrok.connect(PORT)
print('=' * 65)
print(f'🚀 PUBLIC NGROK URL: {tunnel.public_url}')
print(f'🔗 Connect using: python chat.py {tunnel.public_url}')
print('=' * 65)

# Run uvicorn server directly within the running notebook event loop
config = uvicorn.Config(app=app, host='0.0.0.0', port=PORT, log_level='info')
server = uvicorn.Server(config)
await server.serve()